# Sentiment Analysis with Classical ML Models

In this notebook I aim to train and compare various classical machine learning models for sentiment analysis on a text dataset.

In [1]:

!pip uninstall -y protobuf tensorflow-metadata
!pip install -q --upgrade tensorflow-metadata
!pip install -q --upgrade tensorflow-datasets
!pip install -q scikit-learn nltk

Found existing installation: protobuf 5.29.6
Uninstalling protobuf-5.29.6:
  Successfully uninstalled protobuf-5.29.6
Found existing installation: tensorflow-metadata 1.21.0
Uninstalling tensorflow-metadata-1.21.0:
  Successfully uninstalled tensorflow-metadata-1.21.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.35.1 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.


## 1. Data Loading
We will load the IMDB movie reviews dataset, which is a popular dataset for binary sentiment classification (positive/negative reviews).

In [ ]:
import numpy as np
import os
import re
import tarfile

# Direct download URL for the Large Movie Review Dataset (IMDB)
url = "http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
dataset_path = "aclImdb_v1.tar.gz"
extracted_path = "aclImdb"

if not os.path.exists(extracted_path):
    print("Downloading IMDB dataset...")
    import requests
    response = requests.get(url, stream=True)
    with open(dataset_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print("Extracting IMDB dataset...")
    with tarfile.open(dataset_path, "r:gz") as tar:
        tar.extractall()
    os.remove(dataset_path)
else:
    print("IMDB dataset already downloaded and extracted.")

def load_reviews(path):
    texts = []
    labels = [] # 0 for negative, 1 for positive
    for sentiment in ['pos', 'neg']:
        sentiment_path = os.path.join(path, sentiment)
        label = 1 if sentiment == 'pos' else 0
        for fname in os.listdir(sentiment_path):
            with open(os.path.join(sentiment_path, fname), 'r', encoding='utf-8') as f:
                texts.append(f.read())
            labels.append(label)
    return np.array(texts), np.array(labels)


x_train, y_train = load_reviews(os.path.join(extracted_path, 'train'))
print("Loading testing data...")
x_test, y_test = load_reviews(os.path.join(extracted_path, 'test'))


np.random.seed(42)
shuffled_indices = np.random.permutation(len(x_train))
x_train = x_train[shuffled_indices]
y_train = y_train[shuffled_indices]

shuffled_indices = np.random.permutation(len(x_test))
x_test = x_test[shuffled_indices]
y_test = y_test[shuffled_indices]

print(f"\nTraining samples: {len(x_train)}")
print(f"Test samples: {len(x_test)}")
print("\nFirst 3 training reviews and their labels:")
for i in range(3):
    print(f"Review: {x_train[i][:100]}...") # Show first 100 chars
    print(f"Label: {'Positive' if y_train[i] == 1 else 'Negative'}")
    print("---")

IMDB dataset already downloaded and extracted.
Loading testing data...

Training samples: 25000
Test samples: 25000

First 3 training reviews and their labels:
Review: It isn't always easy to explain what a movie is like, but this time I think I've found it. It remind...
Label: Positive
---
Review: JESSICA: A GHOST STORY is as the name implies a ghost story. The theme is meant to be horror but com...
Label: Negative
---
Review: Highly recommended!!<br /><br />A well written, funny film which will appeal to everyone out there w...
Label: Positive
---


#For Data Preprocessing
This step involves cleaning the text data and transforming it into a numerical format suitable for machine learning algorithms. I used techniques like tokenization, stop word removal, and vectorization (TF-IDF).

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
import string
import re


print("Downloading NLTK data (stopwords, punkt)...")
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True) # Added to resolve the new LookupError
print("NLTK data download complete.")

stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):

    text = text.lower()
    text = re.sub(r'<.*?>', '', text)

    text = ''.join([char for char in text if char not in string.punctuation])

    words = nltk.word_tokenize(text)
    words = [stemmer.stem(word) for word in words if word not in stop_words]
    return ' '.join(words)


x_train_preprocessed = np.array([preprocess_text(text) for text in x_train])
print("Preprocessing training data complete.")

x_test_preprocessed = np.array([preprocess_text(text) for text in x_test])
print("Preprocessing test data complete.")


for i in range(3):
    print(f"Original: {x_train[i][:100]}...")
    print(f"Preprocessed: {x_train_preprocessed[i][:100]}...")

NLTK data download complete.
Preprocessing training data complete.
Preprocessing test data complete.
Original: It isn't always easy to explain what a movie is like, but this time I think I've found it. It remind...
Preprocessed: isnt alway easi explain movi like time think ive found remind two movi trainspot small time crimin s...
Original: JESSICA: A GHOST STORY is as the name implies a ghost story. The theme is meant to be horror but com...
Preprocessed: jessica ghost stori name impli ghost stori theme meant horror come across closer comedya woman come ...
Original: Highly recommended!!<br /><br />A well written, funny film which will appeal to everyone out there w...
Preprocessed: highli recommendeda well written funni film appeal everyon sens humour give go good see independ bri...


# For  Feature Extraction

I used  TF-IDF (Term Frequency-Inverse Document Frequency) to convert the preprocessed text data into numerical feature vectors. TF-IDF is a statistical measure that evaluates how relevant a word is to a document in a collection of documents.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

print("Initializing")
vectorizer = TfidfVectorizer(max_features=10000, min_df=5, max_df=0.8)

print("Fitting TF-IDF Vectorizer on training data and transforming...")
X_train_features = vectorizer.fit_transform(x_train_preprocessed)
print("Transforming test data...")
X_test_features = vectorizer.transform(x_test_preprocessed)

print("\nFeature extraction complete.")
print(f"Shape of training features: {X_train_features.shape}")
print(f"Shape of test features: {X_test_features.shape}")

print(f"Number of features (unique words/n-grams): {len(vectorizer.get_feature_names_out())}")
print("First 10 feature names:")
print(vectorizer.get_feature_names_out()[:10])

Initializing
Fitting TF-IDF Vectorizer on training data and transforming...
Transforming test data...

Feature extraction complete.
Shape of training features: (25000, 10000)
Shape of test features: (25000, 10000)
Number of features (unique words/n-grams): 10000
First 10 feature names:
['007' '010' '10' '100' '1000' '10000' '101' '1010' '11' '110']


# And Now for Model Training

I  trained several classical machine learning models on the TF-IDF features derived from the preprocessed text data.  I focused on models commonly used for text classification, including Logistic Regression, Naive Bayes, Support Vector Machines, and Decision Trees.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
print("Initializing models...")
models = {
    'Logistic Regression': LogisticRegression(random_state=42, solver='liblinear'),
    'Multinomial Naive Bayes': MultinomialNB(),
    'Linear SVC': LinearSVC(random_state=42, dual=True),
    'Decision Tree': DecisionTreeClassifier(random_state=42)
}


model_results = {}



for name, model in models.items():
    print(f"\n--- Training {name} ---")


    model.fit(X_train_features, y_train)


    y_pred = model.predict(X_test_features)


    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1 Score: {f1:.4f}")


    model_results[name] = {
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'classification_report': classification_report(y_test, y_pred, output_dict=True)
    }

print("\nAll models trained and evaluated.")


print("\n--- Model Performance Summary ---")
for name, results in model_results.items():
    print(f"\n{name}:")
    print(f"  Accuracy: {results['accuracy']:.4f}")
    print(f"  Precision: {results['precision']:.4f}")
    print(f"  Recall: {results['recall']:.4f}")
    print(f"  F1 Score: {results['f1_score']:.4f}")

Initializing models...

--- Training Logistic Regression ---
  Accuracy: 0.8772
  Precision: 0.8763
  Recall: 0.8783
  F1 Score: 0.8773

--- Training Multinomial Naive Bayes ---
  Accuracy: 0.8281
  Precision: 0.8547
  Recall: 0.7906
  F1 Score: 0.8214

--- Training Linear SVC ---
  Accuracy: 0.8593
  Precision: 0.8672
  Recall: 0.8486
  F1 Score: 0.8578

--- Training Decision Tree ---
  Accuracy: 0.7104
  Precision: 0.7134
  Recall: 0.7032
  F1 Score: 0.7083

All models trained and evaluated.

--- Model Performance Summary ---

Logistic Regression:
  Accuracy: 0.8772
  Precision: 0.8763
  Recall: 0.8783
  F1 Score: 0.8773

Multinomial Naive Bayes:
  Accuracy: 0.8281
  Precision: 0.8547
  Recall: 0.7906
  F1 Score: 0.8214

Linear SVC:
  Accuracy: 0.8593
  Precision: 0.8672
  Recall: 0.8486
  F1 Score: 0.8578

Decision Tree:
  Accuracy: 0.7104
  Precision: 0.7134
  Recall: 0.7032
  F1 Score: 0.7083
